# Детекция ботов-сборщиков данных

Для каждой `cookie_id` по её событиям внутри суточного окна строится одна строка признаков, по ним обучается ансамбль градиентных бустингов. На выходе `score` от 0 до 1: чем выше, тем вероятнее, что кука принадлежит боту.

- **Метрика:** Precision при Recall ≥ 0.7, официальная реализация из `metric.py`.
- **Валидация:** по времени, 7 фолдов: каждый фолд проверяет один день на модели, обученной на всех предыдущих днях.
- **Запуск:** рядом с ноутбуком лежат папка `data/` и `metric.py`, затем Kernel → Restart & Run All. Результат: `submission.csv` и `requirements.txt`.

Описание подхода, экспериментов и ограничений находится в последнем разделе.

In [1]:
import sys
import numpy as np
import pandas as pd
import sklearn
from sklearn.ensemble import HistGradientBoostingClassifier, RandomForestClassifier
from metric import precision_at_recall, recall_at_fpr

RANDOM_STATE = 42
SESSION_GAP = 30 * 60                                   # пауза длиннее 30 минут = новая сессия
PLATFORM_ALIASES = {'desktop': 'web', 'iphone': 'ios'}  # одна платформа под разными именами
EVENT_TYPES = ['item_view', 'search_results_view', 'photo_swipe', 'seller_page_view',
               'favorite_add', 'contact_phone_show', 'contact_chat_open',
               'contact_message_sent', 'login']

print('Python', sys.version.split()[0], '| numpy', np.__version__,
      '| pandas', pd.__version__, '| scikit-learn', sklearn.__version__)

dates = ['cookie_created_at', 'window_start_ts', 'window_end_ts']
train = pd.read_csv('data/train.csv', parse_dates=dates)
test = pd.read_csv('data/test.csv', parse_dates=dates)
events = pd.read_csv('data/events.csv.gz', parse_dates=['event_ts'])

print('train:', train.shape, '| test:', test.shape, '| events:', events.shape)
print('доля ботов в train:', round(train['target'].mean(), 4))

Python 3.12.0 | numpy 2.4.6 | pandas 3.0.3 | scikit-learn 1.9.0
train: (11091, 5) | test: (4909, 4) | events: (328905, 14)
доля ботов в train: 0.0811


## 1. Проверка данных

Что проверяем и зачем:
- **дубликаты**: повторно записанные события завышают счётчики и дают ложные нулевые паузы;
- **порядок строк**: паузы между событиями можно считать только после сортировки по времени;
- **platform**: одна и та же платформа записана в разном регистре и под разными именами;
- **окно наблюдения**: признаки разрешено считать только по событиям внутри окна; отдельно смотрим, где лежат показы капчи;
- **пропуски**: отличаются ли они у людей и ботов.

In [2]:
print('полных дубликатов событий:', events.duplicated().sum())
print('события отсортированы по времени:', events['event_ts'].is_monotonic_increasing)
print('варианты platform:', sorted(events['platform'].unique()))

win = events.merge(train[['cookie_id', 'window_start_ts', 'window_end_ts', 'target']], on='cookie_id')
in_win = (win['event_ts'] >= win['window_start_ts']) & (win['event_ts'] < win['window_end_ts'])
print('\nсобытий train вне окна:', (~in_win).sum(), 'из', len(win))
print('captcha_shown внутри окна:', (win.loc[in_win, 'event_name'] == 'captcha_shown').sum(),
      '| вне окна:', (win.loc[~in_win, 'event_name'] == 'captcha_shown').sum())

print('\nдоля пропусков внутри окон (0 = люди, 1 = боты):')
print(win[in_win].groupby('target')[['item_category', 'item_location', 'seller_type']]
      .apply(lambda d: d.isna().mean()).round(3))

полных дубликатов событий: 4863
события отсортированы по времени: False
варианты platform: ['ANDROID', 'Android', 'IOS', 'WEB', 'Web', 'android', 'desktop', 'iOS', 'ios', 'iphone', 'web']

событий train вне окна: 40779 из 239215
captcha_shown внутри окна: 0 | вне окна: 7928

доля пропусков внутри окон (0 = люди, 1 = боты):
        item_category  item_location  seller_type
target                                           
0               0.080          0.051        0.388
1               0.069          0.038        0.416


Выводы:
- дубликаты удаляем, события сортируем по куке и времени;
- `platform` приводим к нижнему регистру и склеиваем псевдонимы `desktop` → `web` и `iphone` → `ios`: у них одинаковая доля событий с курсором и одинаковое соотношение у людей и ботов;
- все `captcha_shown` лежат **вне** окон наблюдения. Это реакция антибот-системы после окна, а не поведение пользователя: если бы её не отрезать, модель выучила бы метку из будущего. Фильтр по окну эту утечку исключает;
- доли пропусков у людей и ботов одинаковые, это шум логирования, поэтому отдельных признаков из пропусков не строим.

## 2. События внутри окна и очистка

Одна функция для train и test: оставляет события `window_start_ts <= event_ts < window_end_ts`, чистит их и считает `dt`, паузу в секундах с предыдущим событием той же куки.

In [3]:
def prepare_events(events, meta):
    ev = events.merge(meta[['cookie_id', 'window_start_ts', 'window_end_ts']], on='cookie_id')
    ev = ev[(ev['event_ts'] >= ev['window_start_ts']) & (ev['event_ts'] < ev['window_end_ts'])].copy()
    # регистр и псевдонимы приводим до удаления дублей: 'WEB' и 'web' в одинаковых строках тоже дубль
    ev['platform'] = ev['platform'].str.lower().replace(PLATFORM_ALIASES)
    ev = ev.drop_duplicates()
    ev = ev.sort_values(['cookie_id', 'event_ts']).reset_index(drop=True)
    ev['dt'] = ev.groupby('cookie_id')['event_ts'].diff().dt.total_seconds()
    return ev

ev_tr = prepare_events(events, train)
ev_te = prepare_events(events, test)
print('событий в окнах после чистки: train', len(ev_tr), '| test', len(ev_te))

событий в окнах после чистки: train 195482 | test 88348


## 3. Признаки: одна строка на куку

- **Объём и разнообразие**: число событий, объявлений, категорий, городов, запросов, глубина листания выдачи. Сборщики обходят много категорий и городов подряд.
- **Темп**: медиана, разброс и максимум пауз внутри сессий, число сессий, длительность активности. У ботов медианная пауза около 14 секунд против 47 у людей и почти нет долгих перерывов.
- **Курсор и платформа**: доля web-событий, доля событий с курсором, максимум, среднее и разброс координат. У людей курсор проходит почти весь экран 1920×1080, у ботов остаётся в меньшей области, похожей на окно автоматизированного браузера.
- **User-Agent**: флаги HeadlessChrome, HTTP-библиотек, Linux-десктопа и не браузерного формата. Строка целиком не используется, чтобы модель не запоминала конкретные версии.
- **Доли типов событий** и **возраст куки** на начало окна.

Пропуски: счётчики заполняются нулём (нет событий значит ноль событий). Величины, которые нельзя посчитать, например медиана пауз при одном событии или координаты курсора на телефоне, остаются NaN: бустинг обрабатывает их сам как отдельную ситуацию.

In [4]:
POINTER_FEATS = ['web_share', 'ptr_x_max', 'ptr_x_mean', 'ptr_x_std',
                 'ptr_y_max', 'ptr_y_mean', 'ptr_y_std']

def build_features(ev, meta):
    ua = ev['user_agent'].fillna('')
    ev = ev.assign(
        dt_in=ev['dt'].where(ev['dt'] <= SESSION_GAP),              # паузы внутри сессий
        new_session=ev['dt'].isna() | (ev['dt'] > SESSION_GAP),
        hour=ev['event_ts'].dt.hour,
        is_web=ev['platform'] == 'web',
        has_ptr=ev['pointer_x'].notna(),
        ua_headless=ua.str.contains('Headless'),
        ua_http_lib=ua.str.contains('okhttp|python|curl|requests|wget', case=False),
        ua_linux=ua.str.contains('X11; Linux'),
        ua_not_moz=~ua.str.startswith('Mozilla/'),
    )
    f = ev.groupby('cookie_id').agg(
        n_events=('event_ts', 'size'), ts_min=('event_ts', 'min'), ts_max=('event_ts', 'max'),
        n_items=('item_id', 'nunique'), n_categories=('item_category', 'nunique'),
        n_locations=('item_location', 'nunique'), n_queries=('search_query', 'nunique'),
        max_search_page=('search_page', 'max'), mean_search_page=('search_page', 'mean'),
        n_sessions=('new_session', 'sum'), n_hours=('hour', 'nunique'),
        dt_median=('dt', 'median'), dt_in_median=('dt_in', 'median'),
        dt_in_std=('dt_in', 'std'), dt_in_max=('dt_in', 'max'),
        n_web_events=('is_web', 'sum'), web_share=('is_web', 'mean'),
        ptr_events=('has_ptr', 'sum'), ptr_x_unique=('pointer_x', 'nunique'),
        ptr_x_max=('pointer_x', 'max'), ptr_x_mean=('pointer_x', 'mean'), ptr_x_std=('pointer_x', 'std'),
        ptr_y_max=('pointer_y', 'max'), ptr_y_mean=('pointer_y', 'mean'), ptr_y_std=('pointer_y', 'std'),
        n_platforms=('platform', 'nunique'), n_user_agents=('user_agent', 'nunique'),
        ua_headless=('ua_headless', 'max'), ua_http_lib=('ua_http_lib', 'max'),
        ua_linux=('ua_linux', 'max'), ua_not_moz=('ua_not_moz', 'max'),
    )
    f['span_min'] = (f.pop('ts_max') - f.pop('ts_min')).dt.total_seconds() / 60
    f['items_per_event'] = f['n_items'] / f['n_events']
    f['events_per_session'] = f['n_events'] / f['n_sessions']
    # насколько неровный темп: разброс пауз относительно их медианы; деление на 0 даёт NaN, а не inf
    f['dt_in_cv'] = (f['dt_in_std'] / f['dt_in_median']).replace([np.inf, -np.inf], np.nan)
    # курсор пишется только на web, поэтому долю считаем от web-событий; у мобильных кук будет NaN
    f['ptr_share_web'] = f['ptr_events'] / f['n_web_events'].replace(0, np.nan)

    shares = pd.crosstab(ev['cookie_id'], ev['event_name'], normalize='index')
    f = f.join(shares.reindex(columns=EVENT_TYPES, fill_value=0).add_prefix('share_'))

    # каждая кука из meta получает строку, даже если у неё нет событий в окне
    f = meta[['cookie_id', 'cookie_created_at', 'window_start_ts']].merge(
        f.reset_index(), on='cookie_id', how='left')
    f['cookie_age_h'] = (f['window_start_ts'] - f['cookie_created_at']).dt.total_seconds() / 3600
    f = f.drop(columns=['cookie_created_at', 'window_start_ts'])

    count_cols = [c for c in f.columns if c.startswith(('n_', 'share_', 'ptr_events', 'ua_'))]
    f[count_cols] = f[count_cols].fillna(0).astype(float)
    return f

Xtr = build_features(ev_tr, train)
Xte = build_features(ev_te, test)
ytr = train['target'].values
FEATS = [c for c in Xtr.columns if c != 'cookie_id']
print('признаков:', len(FEATS), '| train', Xtr.shape, '| test', Xte.shape)

признаков: 44 | train (11091, 45) | test (4909, 45)


## 4. Модель

Итоговая модель: ансамбль из 5 `HistGradientBoostingClassifier` с разными `random_state`. При каждом разбиении дерево видит случайную половину признаков (`max_features=0.5`), ответы моделей усредняются. Случайные ошибки отдельных моделей при усреднении гасят друг друга, а общий сигнал остаётся: на выборке, где ботов немного, это заметно стабилизирует ранжирование.

Ранняя остановка выключена: она включается автоматически только на выборках больше 10 000 строк, и без этого модель на валидации и на полном train обучалась бы по-разному.

Для сравнения рядом определены baseline из quickstart (случайный лес на двух признаках) и одиночный бустинг.

In [5]:
class SeedEnsemble:
    """Среднее нескольких бустингов с разными seed: снижает разброс предсказаний."""
    def __init__(self, n_models=5, **params):
        self.n_models, self.params = n_models, params

    def fit(self, X, y):
        self.models = [HistGradientBoostingClassifier(random_state=RANDOM_STATE + i, **self.params).fit(X, y)
                       for i in range(self.n_models)]
        return self

    def predict_proba(self, X):
        return np.mean([m.predict_proba(X) for m in self.models], axis=0)


def make_final_model():
    return SeedEnsemble(5, max_features=0.5, max_iter=500, learning_rate=0.03, early_stopping=False)

def make_single_model():
    return HistGradientBoostingClassifier(max_iter=300, learning_rate=0.05,
                                          early_stopping=False, random_state=RANDOM_STATE)

def make_baseline():
    return RandomForestClassifier(n_estimators=300, min_samples_leaf=3, random_state=RANDOM_STATE)

## 5. Валидация по времени

Тест лежит позже трейна, поэтому случайное разбиение не подходит: оно смешало бы будущее с прошлым. Берём последние 7 дней train: для каждого дня обучаемся на всех предыдущих днях и проверяем на нём.

В одном дне всего 40–70 ботов, и метрика на одном разбиении шумит на 1–3 пункта. Поэтому смотрим среднее по 7 дням и в скольких днях один вариант лучше другого. Вариант принимается, только если он лучше в среднем хотя бы на 1.5 пункта и выигрывает минимум в 5 днях из 7.

Ячейка считается несколько минут.

In [6]:
days = np.sort(train['window_start_ts'].dt.normalize().unique())
day = train['window_start_ts'].dt.normalize()
folds = [((day < d).values, (day == d).values) for d in days[-7:]]
for (tr, va), d in zip(folds, days[-7:]):
    print(str(d)[:10], '| train', tr.sum(), '| valid', va.sum(), '| ботов', ytr[va].sum())

def cv_score(make, cols):
    """P@R≥0.7 и Recall при FPR 1% по каждому из 7 дней."""
    pr, rf = [], []
    for tr, va in folds:
        p = make().fit(Xtr.loc[tr, cols], ytr[tr]).predict_proba(Xtr.loc[va, cols])[:, 1]
        pr.append(precision_at_recall(ytr[va], p))
        rf.append(recall_at_fpr(ytr[va], p))
    return np.array(pr), np.array(rf)

feats_no_ptr = [c for c in FEATS if c not in POINTER_FEATS]
candidates = {
    'baseline: лес, 2 признака':        (make_baseline, ['n_events', 'n_items']),
    'бустинг без курсора и платформы':  (make_single_model, feats_no_ptr),
    'бустинг, все признаки':            (make_single_model, FEATS),
    'ансамбль, все признаки (итог)':    (make_final_model, FEATS),
}
results = {}
print()
for name, (make, cols) in candidates.items():
    pr, rf = results[name] = cv_score(make, cols)
    print(f'{name:34s} P@R0.7 = {pr.mean():.4f} ± {pr.std():.3f} | R@FPR1% = {rf.mean():.4f} | по дням {np.round(pr, 3)}')

wins = (results['ансамбль, все признаки (итог)'][0] > results['бустинг, все признаки'][0]).sum()
print(f'\nансамбль лучше одиночного бустинга в {wins} днях из 7')

2026-04-13 | train 5930 | valid 843 | ботов 53
2026-04-14 | train 6773 | valid 826 | ботов 70
2026-04-15 | train 7599 | valid 801 | ботов 65
2026-04-16 | train 8400 | valid 740 | ботов 60
2026-04-17 | train 9140 | valid 713 | ботов 66
2026-04-18 | train 9853 | valid 606 | ботов 52
2026-04-19 | train 10459 | valid 632 | ботов 42

baseline: лес, 2 признака          P@R0.7 = 0.0969 ± 0.015 | R@FPR1% = 0.0927 | по дням [0.073 0.106 0.114 0.093 0.093 0.118 0.082]
бустинг без курсора и платформы    P@R0.7 = 0.5185 ± 0.128 | R@FPR1% = 0.5343 | по дням [0.392 0.516 0.73  0.525 0.305 0.574 0.588]
бустинг, все признаки              P@R0.7 = 0.7687 ± 0.083 | R@FPR1% = 0.6085 | по дням [0.65  0.79  0.821 0.778 0.887 0.644 0.811]
ансамбль, все признаки (итог)      P@R0.7 = 0.7928 ± 0.029 | R@FPR1% = 0.6116 | по дням [0.76  0.817 0.8   0.792 0.825 0.74  0.816]

ансамбль лучше одиночного бустинга в 5 днях из 7


## 6. Обучение на всём train и сабмит

Итоговый ансамбль обучается на всех днях train. Проверки гарантируют формат: ровно одна строка на каждую куку из test, без пропусков и дубликатов, оценки от 0 до 1.

In [7]:
model = make_final_model().fit(Xtr[FEATS], ytr)
sub = pd.DataFrame({'cookie_id': Xte['cookie_id'],
                    'score': model.predict_proba(Xte[FEATS])[:, 1]})

assert len(sub) == len(test)
assert sub['cookie_id'].is_unique and set(sub['cookie_id']) == set(test['cookie_id'])
assert sub['score'].between(0, 1).all() and sub['score'].notna().all()
sub.to_csv('submission.csv', index=False)
print('submission.csv сохранён:', sub.shape, '| различных значений score:', sub['score'].nunique())

with open('requirements.txt', 'w') as fh:
    fh.write(f'numpy=={np.__version__}\npandas=={pd.__version__}\nscikit-learn=={sklearn.__version__}\n')
print(open('requirements.txt').read())

submission.csv сохранён: (4909, 2) | различных значений score: 4906
numpy==2.4.6
pandas==3.0.3
scikit-learn==1.9.0



## Описание решения

**Задача.** По событиям внутри суточного окна дать каждой куке оценку того, что это бот-сборщик данных. Метрика: Precision при Recall ≥ 0.7.

**Данные и очистка** (раздел 1–2)
- Признаки считаются только по событиям внутри окна куки: `window_start_ts <= event_ts < window_end_ts`.
- Все показы капчи лежат вне окон: это след антибот-системы после окна. Фильтр по окну исключает эту утечку.
- Удалены полные дубликаты событий; `platform` приведена к нижнему регистру, псевдонимы `desktop`/`web` и `iphone`/`ios` склеены до удаления дублей.
- События отсортированы по куке и времени, иначе паузы между событиями считаются неверно.
- Пропуски у людей и ботов распределены одинаково, это шум логирования. Счётчики заполняются нулём, невычислимые величины остаются NaN.

**Признаки** (раздел 3): 44 признака в шести группах: объём и разнообразие просмотров, темп и паузы, курсор и платформа, флаги автоматизации в User-Agent, доли типов событий, возраст куки.

**Модель** (раздел 4): ансамбль из 5 градиентных бустингов из scikit-learn со случайной половиной признаков при каждом разбиении, ответы усредняются.

**Валидация** (раздел 5): по времени, 7 фолдов по одному дню, обучение на всех предыдущих днях. Метрика из официального `metric.py`, дополнительно Recall при доле задетых людей 1%.

**Результаты**
- Полная таблица сравнения на 7 днях находится в выводе раздела 5: baseline из quickstart, бустинг без признаков курсора, одиночный бустинг и итоговый ансамбль.
- Главный прирост дали признаки курсора и платформы: на промежуточной валидации (3 фолда по 2 дня) P@R≥0.7 выросла с 0.504 до 0.765 и была лучше на каждом фолде.
- Ансамбль вместо одиночного бустинга: 0.793 против 0.769 в среднем по 7 дням, лучше в 5 днях из 7, худший день 0.74 против 0.64.
- На скрытом тесте промежуточные версии получили 0.545 (без признаков курсора) и 0.782 (одиночный бустинг с курсором).
- Итоговая модель (ансамбль из 5 бустингов) на скрытом тесте: 0.786.

**Что пробовали и не взяли** (прирост меньше шума или ухудшение на валидации)
- Регулярность поведения: квантили пауз, доля одинаковых пауз, листание выдачи строго по порядку, повторные просмотры, фото на просмотр: 0.553 против 0.543, лучше только на 1 фолде из 3.
- Доли категорий, типов продавцов и платформ: 0.510 против 0.543, модель запоминает популярность категорий в конкретные дни.
- Модель устройства из User-Agent (0.766 против 0.765), паузы перед каждым типом действия (0.763: у людей паузы тоже не зависят от типа действия), популярность объявлений, запросов и User-Agent за день (0.753).
- Признаки сборщиков контактов на мобильных: контакт без входа в аккаунт, ночная активность, частные продавцы (0.739).
- Разнообразие с поправкой на объём: энтропия категорий и городов, новые категории на событие (0.755 против 0.793 у ансамбля без них).
- Удаление технических признаков (возраст куки, число платформ и User-Agent): 0.761 и 0.753 против 0.765, признаки оставлены.
- Другие модели до добавления курсора: более осторожные деревья 0.518, балансировка классов 0.517, случайный лес 0.470 против 0.543.
- Проверки без сигнала: события ровно в :00 секунд, неподвижный курсор, версии приложения и Android.

**Ограничения**
- На валидации около 400 ботов за 7 дней: различия меньше 1.5–2 пунктов неотличимы от шума.
- Самое слабое место: мобильные куки. Курсора на телефоне нет, и при общем пороге модель находит там около 55% ботов против 77% на web.
- Курсор и User-Agent это технические следы инструмента автоматизации. Если сервис начнёт запускать браузер в полноэкранном окне или имитировать курсор, качество упадёт, останутся поведенческие признаки темпа и разнообразия.
- Разметка построена на известных сервисах сбора данных, качество на новых, неизвестных ботах может быть ниже.

**Воспроизводимость:** Python 3.12, версии библиотек в `requirements.txt` (numpy, pandas, scikit-learn; всё open-source, запускается локально, внешние API и языковые модели в решении не используются). Все `random_state` зафиксированы, ранняя остановка выключена. Запуск: Kernel → Restart & Run All, результат `submission.csv`.